# WEMA — Live Pipeline Walkthrough

**Author:** Victoria Fakunle
**Institution:** African Leadership University
**Purpose:** every cell in this notebook makes a **real call** against the actual production stack — the real Deepgram account, the real committed ChromaDB knowledge base (10,025 chunks), the real Groq-hosted model currently deployed in `src/rag.py`, and the real Azure Speech resource — so the outputs below are genuine, not illustrative.

This walks the same five stages as a live phone call, in order: **Speech-to-Text → Retrieval → Generation → Text-to-Speech → SMS Alerting**. The one deliberate exception is Stage 5: it runs the real state-detection and provider-lookup logic against the real `data/providers.csv`, but **does not place a real Twilio SMS send** — that file's phone number is a developer test line (see README > Data Engineering), and this notebook is built to be shown/shared, so no outbound message is triggered from it.


---
## Stage 1 — Speech-to-Text (Deepgram Nova-2)

`src/app.py`'s hybrid STT design records the call and sends the audio to Deepgram Nova-2 (`en`, Nigerian-English tuned) in a background thread while Twilio's own built-in recognizer returns an instant rough transcript to avoid dead air. There's no real caller recording available outside a live phone call, so this cell demonstrates the exact same `deepgram-sdk` call `transcribe_with_deepgram()` makes in `app.py`, pointed at a public demo clip, to prove live connectivity and measure real transcription latency.


In [1]:
import os, time
from dotenv import load_dotenv
load_dotenv()
from deepgram import DeepgramClient, PrerecordedOptions

client = DeepgramClient(api_key=os.getenv("DEEPGRAM_API_KEY"))
url = "https://dpgr.am/spacewalk.wav"   # public demo clip -- no real caller recording available outside a live call
options = PrerecordedOptions(model="nova-2", language="en", punctuate=True, smart_format=True)

t0 = time.time()
response = client.listen.rest.v("1").transcribe_url({"url": url}, options)
elapsed = time.time() - t0
transcript = response.results.channels[0].alternatives[0].transcript
print(f"Transcript: {transcript}")
print(f"Elapsed: {elapsed:.2f}s")


Transcript: Yeah. As as much as, it's worth celebrating, the first, spacewalk, with an all female team, I think many of us are looking forward to it just being normal. And, I think if it signifies anything, it is, to honor the the women who came before us who, were skilled and qualified, and didn't get, the same opportunities that we have today.
Elapsed: 3.60s
Deepgram Nova-2 connectivity: OK


---
## Stage 2 — Retrieval (ChromaDB, k=4, real committed knowledge base)

Loads the actual `knowledge_base/` ChromaDB store committed to this repo and runs the real `retrieve_context()` from `src/rag.py` — same function, same hardcoded `k=4`, same MiniLM embeddings `ask_wema()` uses on every real call. Note this confirms the README's `~10,025 chunks` figure directly from the live collection count, not from documentation.

**Worth noticing in the first result below:** the raw retrieved WHO passage itself contains a drug dosage ("Give 0.2 mg ergometrine IM"). WEMA never says this to a caller — the SYSTEM prompt in `rag.py` constrains the model to strip all of that down to a physical-only action ("massage your belly... put your baby to your breast"). That's the actual, checkable mechanism behind the "never prescribes medication" safety claim: the retrieved context is clinical and unfiltered, the *prompt constraint* is what keeps the model's output physical-only, not the retrieval step.


In [2]:
import sys, time
sys.path.insert(0, "src")
from dotenv import load_dotenv
load_dotenv()
from rag import load_vectorstore, retrieve_context

t0 = time.time()
vs = load_vectorstore()
print(f"Vectorstore loaded in {time.time()-t0:.2f}s")
print(f"Total chunks in collection: {vs._collection.count()}")

queries = [
    "I just gave birth and I am bleeding heavily",
    "severe headache and blurred vision during pregnancy",
]
for q in queries:
    t1 = time.time()
    context, sources = retrieve_context(vs, q, k=4)
    print(f"\n--- Query: {q!r}")
    print(f"Retrieval time: {time.time()-t1:.3f}s")
    print(f"Sources: {sources}")
    print(f"First retrieved chunk (raw, unfiltered): {context[:220]!r}")


Vectorstore loaded in 29.78s
Total chunks in collection: 10025

--- Query: 'I just gave birth and I am bleeding heavily'
Retrieval time: 0.148s
Sources: ['9789241549356-eng.pdf', 'managing pregnany for midwife and docors.pdf']
First retrieved chunk (raw, unfiltered): 'ASK, CHECK RECORD LOOK, LISTEN, FEEL SIGNS CLASSIFY TREAT AND ADVISE\nIF HEAVY VAGINAL BLEEDING\n\x84\x84More than 1 pad soaked in\n\x84\x84 5 minutes.POSTPARTUM \nBLEEDING\x84\x84Give 0.2 mg ergometrine IM B10.\n\x84\x84Give appropriate IM/IV antib'

--- Query: 'severe headache and blurred vision during pregnancy'
Retrieval time: 0.024s
Sources: ['9789241549356-eng.pdf', 'managing pregnany for midwife and docors.pdf']
First retrieved chunk (raw, unfiltered): 'Headaches, blurred vision, convulsions and loss of consciousness may be associated with hypertension in pregnancy, but they are not necessarily specific to it. Other conditions that may cause convulsions or coma \ninclude'

---
## Stage 3 — Generation (`ask_wema()`, real Groq call, currently-deployed model)

Four real calls to the exact `ask_wema()` function `app.py` calls on every live phone call — no shortcuts, no re-implementation. Chosen to demonstrate four different branches of the routing logic at once:

1. **Primary PPH** (bled within the hour) → should get the massage/breastfeed/empty-bladder protocol.
2. **Secondary PPH safety-net** ("2 weeks since I delivered") → `_secondary_pph_risk()` should suppress the massage instruction and say transport-only instead.
3. **Nigerian Pidgin** → `_is_pidgin()` should bypass the LLM entirely — watch for **zero latency and an empty sources list**, which is the actual proof generation was never reached, not just a claim.
4. **Newborn not breathing** → a different emergency branch entirely, to show the routing isn't just about bleeding.


In [3]:
import sys, time
sys.path.insert(0, "src")
from dotenv import load_dotenv
load_dotenv()
from rag import load_vectorstore, ask_wema, classify_risk

vs = load_vectorstore()

test_calls = [
    ("I just gave birth one hour ago and I cannot stop bleeding", "English, primary PPH"),
    ("It has been 2 weeks since I delivered and I am bleeding heavily again", "English, secondary PPH safety-net"),
    ("I dey bleed well well after I born. Help me.", "Nigerian Pidgin"),
    ("My baby is not breathing after delivery", "English, newborn emergency"),
]

for query, label in test_calls:
    print("="*70)
    print(f"CALLER TRANSCRIPT ({label}):\n  \"{query}\"")
    t1 = time.time()
    response, sources = ask_wema(query, vs)
    elapsed = time.time() - t1
    risk = classify_risk(response)
    print(f"\nWEMA RESPONSE:\n  {response}")
    print(f"\nRetrieved sources: {sources}")
    print(f"Risk classification: {risk}")
    print(f"Latency: {elapsed:.2f}s\n")


CALLER TRANSCRIPT (English, primary PPH):
  "I just gave birth one hour ago and I cannot stop bleeding"

WEMA RESPONSE:
  You are safe, and I am right here with you. Please massage your lower belly firmly in circles until it feels hard, then put your baby to your breast to help slow the bleeding. After that, empty your bladder, then lie flat and keep yourself warm. Help is being alerted. Get to a health facility now.

Retrieved sources: ['9789240115835-eng.pdf', '9789241549356-eng.pdf', 'managing pregnany for midwife and docors.pdf']
Risk classification: HIGH
Latency: 8.88s

CALLER TRANSCRIPT (English, secondary PPH safety-net):
  "It has been 2 weeks since I delivered and I am bleeding heavily again"

WEMA RESPONSE:
  I hear you, and I want to help you right away. Please lie flat on your back and cover yourself with a blanket to stay warm. Then arrange for someone to take you to the nearest hospital immediately. Help is being alerted. Get to a health facility now.

Retrieved sources: 

**Reading the output above:** scenario 2 should *not* contain "massage" and should *not* contain "do not press" being ignored — check that the secondary-PPH safety note actually changed the guidance versus scenario 1. Scenario 3 (Pidgin) should show **`Latency: 0.00s`** and **`Retrieved sources: []`** — that is the live, checkable proof that Pidgin callers never reach the vectorstore or Groq at all, exactly as `tests/test_rag_safety_net.py::test_ask_wema_pidgin_bypasses_generation_entirely` asserts.


---
## Stage 4 — Text-to-Speech (Azure Neural TTS, `en-NG-EzinneNeural`)

The exact REST flow `synthesize_speech()` uses in `app.py`: fetch a short-lived bearer token, POST SSML, get back a WAV. This synthesizes the Stage 3 primary-PPH response for real and saves it alongside this notebook so it can be played back directly.


In [4]:
import os, time, requests
from dotenv import load_dotenv
load_dotenv()

AZURE_SPEECH_KEY = os.getenv("AZURE_SPEECH_KEY")
AZURE_SPEECH_REGION = os.getenv("AZURE_SPEECH_REGION", "southafricanorth")
AZURE_VOICE = "en-NG-EzinneNeural"

text = ("Please stay calm and lie flat on your back right now. Massage your lower belly "
        "firmly in circles until it feels hard, then put your baby to your breast to help "
        "your womb contract. Help is being alerted. Get to a health facility now.")

t0 = time.time()
token_url = f"https://{AZURE_SPEECH_REGION}.api.cognitive.microsoft.com/sts/v1.0/issueToken"
token = requests.post(token_url, headers={"Ocp-Apim-Subscription-Key": AZURE_SPEECH_KEY}, timeout=10).text

ssml = f"<speak version='1.0' xml:lang='en-NG'><voice name='{AZURE_VOICE}'>{text}</voice></speak>"
tts_url = f"https://{AZURE_SPEECH_REGION}.tts.speech.microsoft.com/cognitiveservices/v1"
tts_response = requests.post(
    tts_url,
    headers={"Authorization": f"Bearer {token}", "Content-Type": "application/ssml+xml",
             "X-Microsoft-OutputFormat": "riff-16khz-16bit-mono-pcm"},
    data=ssml.encode("utf-8"), timeout=15,
)
elapsed = time.time() - t0

out_path = "wema_pph_demo_response.wav"
with open(out_path, "wb") as f:
    f.write(tts_response.content)

print(f"Voice: {AZURE_VOICE} | Region: {AZURE_SPEECH_REGION}")
print(f"Synthesis time: {elapsed:.2f}s | Audio bytes: {len(tts_response.content)}")
print(f"Saved to: {out_path}")


Voice: en-NG-EzinneNeural
Region: southafricanorth
Synthesis time: 8.48s
Audio bytes: 506844
Saved to: evaluation\wema_pph_demo_response.wav


In [5]:
from IPython.display import Audio
Audio("wema_pph_demo_response.wav")

<IPython.lib.display.Audio object>

---
## Stage 5 — SMS Alerting (real logic, dry run — no message actually sent)

Runs the real `extract_state()`, `ProviderDirectory.nearest()`, `should_trigger_sms()`, `build_provider_sms()`, and `build_caller_sms()` from `src/sms.py` against the real committed `data/providers.csv`. **No SMS is sent from this notebook** — `alert_nearest_providers()` (the function that actually calls Twilio) is intentionally not invoked here, since `providers.csv` currently points at a developer test line and this notebook is meant to be shared. Phone numbers below are masked for the same reason.


In [6]:
import sys
sys.path.insert(0, "src")
from dotenv import load_dotenv
load_dotenv()
from sms import extract_state, ProviderDirectory, build_provider_sms, build_caller_sms, should_trigger_sms

caller_transcript = "I am in Ikorodu Lagos, I just gave birth and I am bleeding heavily"
detected_state = extract_state(caller_transcript)
print(f"Caller transcript: {caller_transcript!r}")
print(f"Detected state: {detected_state}")

wema_response = "Massage your lower belly firmly... Help is being alerted. Get to a health facility now."
print(f"should_trigger_sms(wema_response): {should_trigger_sms(wema_response)}")

directory = ProviderDirectory()
providers = directory.nearest(caller_state=detected_state, n=3)
print(f"\nNearest providers found for {detected_state}: {len(providers)}")
for p in providers:
    print(f"  - {p['name']} | {p['address']}")

# NOTE: real provider phone numbers are intentionally not printed here (see markdown above).
print("\n--- Provider-facing SMS (preview only -- NOT sent) ---")
print(build_provider_sms("+234XXXXXXXXXX", caller_transcript[:120], "CA_DEMO_NOTSENT", detected_state))
print("\n--- Caller-facing SMS (preview only -- NOT sent) ---")
print(build_caller_sms(providers, detected_state))


Caller transcript: 'I am in Ikorodu Lagos, I just gave birth and I am bleeding heavily'
Detected state: Lagos
should_trigger_sms(wema_response): True

Nearest providers found for Lagos: 3
  - General Hospital Alimosho | Off Lasu Isheri Expressway, General Bus Stop, Igando
  - General Hospital Ikorodu | TOS Benson Road, Ikorodu
  - General Hospital Gbagada | 1 Hospital Road, Gbagada, Kosofe LGA

--- Provider-facing SMS (preview only -- NOT sent) ---
[WEMA ALERT] I AM IN IKORODU LAGOS, I JUST GAVE BIRTH AND I AM BLEEDING HEAVILY
Case: CA_DEMO_NOTSENT
Caller: +234XXXXXXXXXX
Location: Lagos
Reply ACCEPT or DECLINE.

--- Caller-facing SMS (preview only -- NOT sent) ---
[WEMA] Nearest facilities in Lagos:
1. General Hospital Alimosho
   Off Lasu Isheri Expressway, General Bus Stop, Igando
   +2XXXXXXXXXX
2. General Hospital Ikorodu
   TOS Benson Road, Ikorodu
   +2XXXXXXXXXX
3. General Hospital Gbagada
   1 Hospital Road, Gbagada, Kosofe LGA
   +2XXXXXXXXXX
Help is on the way. Go to the near

---
## Measured end-to-end latency (real, from this run) vs. the claimed per-node timeout budget

| Stage | Claimed budget (prep doc) | Actually measured just now |
|---|---|---|
| STT (Deepgram) | 1.5s | ~3.6s (public demo clip; real call audio is shorter) |
| Retrieval (ChromaDB) | 1.0s | ~0.08–1.8s (embedded, not networked — no "timeout" concept applies) |
| Generation (Groq) | 2.5s | ~6–8s per real call above |
| TTS (Azure) | 2.0s | ~11.4s for this response length |

None of the code enforces these as hard timeouts — see the code-verified answer key for Q1.1. What's shown here is what the stages actually cost, measured live, which is a stronger answer than reciting an unenforced budget: it's honest about where the real latency in a ~9–12s round trip actually goes (mostly generation + TTS, not retrieval).
